# Stage 2: Demand Models

**Goal of this step:** We train two separate linear regression models — one on loyal clients, one on non-loyal clients — then combine them with the loyalty matrix from the previous step to build the 200x8 demand matrix that feeds the optimizer.

>Note: Functions are in `handlers/demand_handlers.py`. Distance computation and data loading are reused from `handlers/classifier_handlers.py`.

In [ ]:
import sys
sys.path.append('handlers')
import numpy as np
import matplotlib.pyplot as plt
import demand_handlers as dh

historic, potential, sites = dh.load_data()
loyalty_matrix = np.load('loyalty_matrix.npy')

print(f"Historic: {len(historic)} clients | Potential: {len(potential)} clients | Sites: {len(sites)}")
print(f"Loyalty matrix: {loyalty_matrix.shape} — {loyalty_matrix.mean():.1%} predicted loyal")

## 1. Split historic data by loyalty group

In [ ]:
loyal, nonloyal = dh.split_by_loyalty(historic)
print(f"Loyal group    : {len(loyal)} clients")
print(f"Non-loyal group: {len(nonloyal)} clients")

splits = dh.get_splits(loyal, nonloyal)

## 2. Train the models

We use linear regression on each group separately. The case expects demand to decrease linearly with distance, so this is the natural choice. Negative predictions will be clipped to 0 when building the demand matrix.

In [ ]:
models = dh.train_demand_models(splits)
print("Models trained:", list(models.keys()))

## 3. Evaluation

In [ ]:
dh.evaluation_report(models, splits)

In [ ]:
dh.plot_predictions_vs_actual(models, splits)
plt.show()

## 4. Build the demand matrix

For each (client, site) pair we compute the distance, look up the predicted loyalty from the loyalty matrix, and apply the matching model. => this gives us a 200x8 demand matrix ready for the optimizer.

In [ ]:
demand_matrix = dh.build_demand_matrix(models, loyalty_matrix, potential, sites)

print(f"Demand matrix shape : {demand_matrix.shape}")
print(f"Overall mean demand : {demand_matrix.mean():.2f}")
print(f"Zero-demand entries : {(demand_matrix == 0).sum()} ({(demand_matrix == 0).mean():.1%})")

In [ ]:
dh.plot_demand_matrix_summary(demand_matrix)
plt.show()

In [ ]:
# Save for the optimization step
np.save('demand_matrix.npy', demand_matrix)
print('Saved: demand_matrix.npy')